# 11002BC Hybrid: All-Variant Model-Family Benchmark, ver01

This notebook compares multiple learner families across every completed Hybrid variant whose feature-ranking artifact is available in the selected Part B run.

It does not regenerate raw per-engine features. It reconstructs one in-memory feature matrix from completed caches, loads each Hybrid variant's ranked feature list, applies identical grouped folds, and evaluates every available model family.

Core controls:

- Aircraft-grouped cross-validation
- Same target, rows, folds, and training-row policy across model families
- Variant-specific top-500 and top-1,000 feature lists when available
- Optional LightGBM, XGBoost, and CatBoost discovery
- One model at a time, with explicit cleanup
- JSON-safe results and restartable per-model result files
- Fold-local preprocessing for linear models

This remains exploratory when a feature-ranking file was produced using all labels. Promotion-quality validation requires fold-local feature ranking or a locked out-of-time holdout.


## 1. Configuration and package discovery

The benchmark reuses the newest existing Part B run. Set `TRAIN_ROW_CAP = None` for uncapped training. The default cap keeps large tree ensembles practical while scoring every validation row.


In [1]:
# =============================================================================
# CONFIGURATION
# =============================================================================
from pathlib import Path
from datetime import datetime, timezone
import gc
import importlib.util
import json
import os
import re
import time
import traceback
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=FutureWarning)

WORK_ROOT = Path("/raid3/e296408/All_ECFRs_working")
BC_ROOT = WORK_ROOT / "ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED"
ROW_V2_ROOT = WORK_ROOT / "ECFR_11001_ROW_V2" / "records"
ENGINE_ROOT = BC_ROOT / "engines"
DATA_RICHNESS_ROOT = BC_ROOT / "data_richness"

runs = sorted(DATA_RICHNESS_ROOT.glob("run_*"), key=lambda p: p.stat().st_mtime)
if not runs:
    raise RuntimeError(f"No completed Part B run found under {DATA_RICHNESS_ROOT}")
PART_B_RUN_ROOT = runs[-1]
DATA_ROOT = PART_B_RUN_ROOT / "data"
MODEL_INPUT_FILE = DATA_ROOT / "landing_level_model_inputs.parquet"
if not MODEL_INPUT_FILE.is_file():
    raise FileNotFoundError(MODEL_INPUT_FILE)

OUTPUT_ROOT = PART_B_RUN_ROOT / "hybrid_all_variants_model_family_benchmark_ver01"
RESULT_ROOT = OUTPUT_ROOT / "per_model_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 11002
N_SPLITS = 5
N_JOBS = min(50, os.cpu_count() or 8)
TRAIN_ROW_CAP = 500_000       # Set to None to train every learner on every training row.
TOP_FEATURE_CAPS = (500, 1000)
HORIZON_DAYS = 14
FEATURE_CUTOFF_SECONDS = 900
MIN_TS = pd.Timestamp("2000-01-01", tz="UTC")
MAX_TS = pd.Timestamp(datetime.now(timezone.utc))

PACKAGE_AVAILABILITY = {
    "lightgbm": importlib.util.find_spec("lightgbm") is not None,
    "xgboost": importlib.util.find_spec("xgboost") is not None,
    "catboost": importlib.util.find_spec("catboost") is not None,
}

print({
    "part_b_run_root": str(PART_B_RUN_ROOT),
    "model_input": str(MODEL_INPUT_FILE),
    "output_root": str(OUTPUT_ROOT),
    "workers": N_JOBS,
    "train_row_cap": TRAIN_ROW_CAP,
    "optional_packages": PACKAGE_AVAILABILITY,
})

{'part_b_run_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z', 'model_input': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/data/landing_level_model_inputs.parquet', 'output_root': '/raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_all_variants_model_family_benchmark_ver01', 'workers': 50, 'train_row_cap': 500000, 'optional_packages': {'lightgbm': False, 'xgboost': True, 'catboost': False}}


## 2. Discover completed Hybrid experiments and ranking artifacts

Every `feature_ranking_*.csv` under the current Part B run is cataloged. Variant identity comes from the experiment directory and ranking filename, so earlier and later Hybrid branches can coexist.


In [2]:
# =============================================================================
# DISCOVER RANKINGS AND FEATURE CACHES
# =============================================================================
ranking_files = sorted(PART_B_RUN_ROOT.glob("hybrid*/feature_ranking_*.csv"))
if not ranking_files:
    raise RuntimeError("No Hybrid feature-ranking files were found")

ranking_catalog = []
for path in ranking_files:
    try:
        frame = pd.read_csv(path, nrows=5)
        if "feature" not in frame.columns:
            continue
        ranking_catalog.append({
            "experiment": path.parent.name,
            "ranking_file": path.name,
            "path": str(path),
            "modified_ns": path.stat().st_mtime_ns,
        })
    except Exception:
        continue
ranking_catalog = pd.DataFrame(ranking_catalog).sort_values(
    ["experiment", "ranking_file", "modified_ns"]
).reset_index(drop=True)
ranking_catalog.to_csv(OUTPUT_ROOT / "ranking_catalog.csv", index=False)
print(ranking_catalog.to_string(index=False))

# Shared and experiment-local cache directories containing engine-keyed Parquet files.
candidate_cache_roots = []
for root in [PART_B_RUN_ROOT / "_shared_feature_cache", PART_B_RUN_ROOT]:
    if not root.exists():
        continue
    for directory in root.rglob("*"):
        if directory.is_dir() and next(directory.glob("engine=*.parquet"), None) is not None:
            candidate_cache_roots.append(directory)
# Prefer shallow, shared, newest cache roots. Duplicate schemas are resolved later.
candidate_cache_roots = sorted(set(candidate_cache_roots), key=lambda p: ("_shared_feature_cache" not in str(p), len(p.parts), -p.stat().st_mtime_ns))
print({"ranking_files": len(ranking_catalog), "candidate_cache_roots": len(candidate_cache_roots)})

                                experiment                     ranking_file                                                                                                                                                                                     path         modified_ns
            hybrid_v2_cross_side_benchmark    feature_ranking_hybrid_v1.csv               /raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_v2_cross_side_benchmark/feature_ranking_hybrid_v1.csv 1789982052093245367
            hybrid_v2_cross_side_benchmark    feature_ranking_hybrid_v2.csv               /raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data_richness/run_20260920_120031Z/hybrid_v2_cross_side_benchmark/feature_ranking_hybrid_v2.csv 1789982208521240529
hybrid_v3_2_all_fault_landing_memory_ver05    feature_ranking_hybrid_v3.csv   /raid3/e296408/All_ECFRs_working/ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/data

## 3. Load the base landing table and completed feature caches

Only columns needed by at least one discovered ranking are loaded. Cache columns are joined by `engine_channel_landing_id`; duplicate feature names are skipped after the first preferred source.


In [3]:
# =============================================================================
# LOAD BASE TABLE AND ONLY REQUIRED CACHE FEATURES
# =============================================================================
base = pd.read_parquet(MODEL_INPUT_FILE)
base["engine_channel_landing_id"] = base["engine_channel_landing_id"].astype(str)
base["landing_datetime_utc"] = pd.to_datetime(base["landing_datetime_utc"], errors="coerce", utc=True)
base = base.loc[
    base["landing_datetime_utc"].notna()
    & base["landing_datetime_utc"].between(MIN_TS, MAX_TS)
].reset_index(drop=True)
base["feature_cutoff_utc"] = base["landing_datetime_utc"] + pd.to_timedelta(FEATURE_CUTOFF_SECONDS, unit="s")

ranked_features = set()
for path_text in ranking_catalog["path"]:
    rank = pd.read_csv(path_text, usecols=["feature"])
    ranked_features.update(rank["feature"].dropna().astype(str))

required_identifiers = {
    "engine_channel_landing_id", "engine_serial", "aircraft_identifier",
    "aircraft_landing_event_id", "cross_side_engine_serial", "landing_datetime_utc",
    "feature_cutoff_utc",
}
keep_base = [c for c in base.columns if c in ranked_features or c in required_identifiers]
enriched = base[keep_base].copy()
loaded_features = set(enriched.columns)
cache_audit = []

for cache_root in candidate_cache_roots:
    files = sorted(cache_root.glob("engine=*.parquet"))
    if not files:
        continue
    try:
        schema = set(pq.ParquetFile(files[0]).schema_arrow.names)
    except Exception:
        continue
    wanted = sorted((schema & ranked_features) - loaded_features)
    if not wanted or "engine_channel_landing_id" not in schema:
        continue
    frames = []
    for file in files:
        try:
            names = set(pq.ParquetFile(file).schema_arrow.names)
            cols = ["engine_channel_landing_id"] + [c for c in wanted if c in names]
            frames.append(pd.read_parquet(file, columns=cols))
        except Exception:
            continue
    if not frames:
        continue
    cache_frame = pd.concat(frames, ignore_index=True, sort=False)
    cache_frame["engine_channel_landing_id"] = cache_frame["engine_channel_landing_id"].astype(str)
    cache_frame = cache_frame.drop_duplicates("engine_channel_landing_id", keep="first")
    before = len(enriched)
    enriched = enriched.merge(cache_frame, on="engine_channel_landing_id", how="left", validate="one_to_one")
    if len(enriched) != before:
        raise RuntimeError(f"Cache join changed row count: {cache_root}")
    actually_loaded = [c for c in wanted if c in enriched.columns]
    loaded_features.update(actually_loaded)
    cache_audit.append({"cache_root": str(cache_root), "files": len(files), "features_loaded": len(actually_loaded)})
    del frames, cache_frame
    gc.collect()

pd.DataFrame(cache_audit).to_csv(OUTPUT_ROOT / "cache_load_audit.csv", index=False)
print({
    "rows": len(enriched),
    "base_columns": len(keep_base),
    "ranked_features_requested": len(ranked_features),
    "ranked_features_loaded_directly": len(loaded_features & ranked_features),
    "cache_roots_used": len(cache_audit),
})

{'rows': 1669294, 'base_columns': 6088, 'ranked_features_requested': 10692, 'ranked_features_loaded_directly': 7593, 'cache_roots_used': 6}


## 4. Reconstruct cross-side and delta features when rankings require them

For a requested `XSIDE_`, `DELTA_`, or `ABSDELTA_` feature whose focal source exists, this cell reconstructs the feature at the same aircraft landing event. Existing persisted columns are not duplicated.


In [4]:
# =============================================================================
# RECONSTRUCT REQUESTED CROSS-SIDE / DELTA FEATURES
# =============================================================================
for col in ["aircraft_landing_event_id", "engine_serial", "cross_side_engine_serial"]:
    if col not in enriched.columns:
        raise RuntimeError(f"Missing cross-side identifier: {col}")
    enriched[col] = enriched[col].astype("string").str.strip()

prefixes = ("XSIDE_", "DELTA_", "ABSDELTA_")
requested_derived = sorted(c for c in ranked_features if c.startswith(prefixes) and c not in enriched.columns)
focal_sources = set()
for feature in requested_derived:
    if feature.startswith("ABSDELTA_"):
        focal_sources.add(feature[len("ABSDELTA_"):])
    elif feature.startswith("DELTA_"):
        focal_sources.add(feature[len("DELTA_"):])
    elif feature.startswith("XSIDE_"):
        focal_sources.add(feature[len("XSIDE_"):])
focal_sources = sorted(c for c in focal_sources if c in enriched.columns)

if focal_sources:
    lookup = enriched[["aircraft_landing_event_id", "engine_serial"] + focal_sources].drop_duplicates(
        ["aircraft_landing_event_id", "engine_serial"], keep="first"
    ).rename(columns={"engine_serial": "cross_side_engine_serial", **{c: f"__PARTNER__{c}" for c in focal_sources}})
    enriched = enriched.merge(
        lookup,
        on=["aircraft_landing_event_id", "cross_side_engine_serial"],
        how="left",
        validate="many_to_one",
    )
    for source in focal_sources:
        partner = f"__PARTNER__{source}"
        if f"XSIDE_{source}" in ranked_features and f"XSIDE_{source}" not in enriched.columns:
            enriched[f"XSIDE_{source}"] = pd.to_numeric(enriched[partner], errors="coerce").astype("float32")
        if f"DELTA_{source}" in ranked_features and f"DELTA_{source}" not in enriched.columns:
            enriched[f"DELTA_{source}"] = (
                pd.to_numeric(enriched[source], errors="coerce") - pd.to_numeric(enriched[partner], errors="coerce")
            ).astype("float32")
        if f"ABSDELTA_{source}" in ranked_features and f"ABSDELTA_{source}" not in enriched.columns:
            enriched[f"ABSDELTA_{source}"] = (
                pd.to_numeric(enriched[source], errors="coerce") - pd.to_numeric(enriched[partner], errors="coerce")
            ).abs().astype("float32")
    enriched.drop(columns=[f"__PARTNER__{c}" for c in focal_sources], inplace=True)
    del lookup
    gc.collect()

print({
    "requested_derived_features": len(requested_derived),
    "focal_sources_available": len(focal_sources),
    "derived_features_now_available": sum(c in enriched.columns for c in requested_derived),
})

/tmp/ipykernel_187330/262183159.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched[f"XSIDE_{source}"] = pd.to_numeric(enriched[partner], errors="coerce").astype("float32")
/tmp/ipykernel_187330/262183159.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched[f"ABSDELTA_{source}"] = (
/tmp/ipykernel_187330/262183159.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd

{'requested_derived_features': 3020, 'focal_sources_available': 992, 'derived_features_now_available': 2791}


## 5. Build the common future Fault 206 target

The target uses corrected `FF_ATTRIBUTES` Fault 206 events strictly after the feature cutoff and within 14 days.


In [5]:
# =============================================================================
# REBUILD COMMON FF206 TARGET
# =============================================================================
parts = sorted(ROW_V2_ROOT.glob("file_hash=*/part-*.parquet"))
selected_engines = set(enriched["engine_serial"].astype(str))
ff_frames = []
for index, path in enumerate(parts, 1):
    schema = set(pq.ParquetFile(path).schema_arrow.names)
    required = {"engine_serial", "section", "fault_code", "record_datetime_utc"}
    if not required.issubset(schema):
        continue
    frame = pd.read_parquet(path, columns=sorted(required))
    mask = (
        frame["engine_serial"].astype(str).isin(selected_engines)
        & frame["section"].fillna("").astype(str).str.upper().eq("FF_ATTRIBUTES")
        & pd.to_numeric(frame["fault_code"], errors="coerce").eq(206)
    )
    frame = frame.loc[mask, ["engine_serial", "record_datetime_utc"]].copy()
    if not frame.empty:
        frame["fault206_datetime_utc"] = pd.to_datetime(frame["record_datetime_utc"], errors="coerce", utc=True)
        ff_frames.append(frame[["engine_serial", "fault206_datetime_utc"]])
    if index == 1 or index % 10000 == 0 or index == len(parts):
        print(f"[TARGET {index:,}/{len(parts):,}] matching partitions={len(ff_frames):,}")
if not ff_frames:
    raise RuntimeError("No corrected FF_ATTRIBUTES Fault 206 rows found")
fault206 = pd.concat(ff_frames, ignore_index=True).dropna().drop_duplicates()
fault_times = {
    str(engine): np.sort(group["fault206_datetime_utc"].astype("int64").to_numpy(dtype=np.int64))
    for engine, group in fault206.groupby("engine_serial", sort=False)
}
cutoff_ns = enriched["feature_cutoff_utc"].astype("int64").to_numpy(dtype=np.int64)
horizon_ns = np.int64(pd.Timedelta(days=HORIZON_DAYS).value)
y = np.zeros(len(enriched), dtype=np.int8)
for i, (engine, cutoff) in enumerate(zip(enriched["engine_serial"].astype(str), cutoff_ns)):
    times = fault_times.get(engine)
    if times is None:
        continue
    position = np.searchsorted(times, cutoff, side="right")
    if position < len(times):
        y[i] = np.int8(times[position] <= cutoff + horizon_ns)
TARGET = "FAULT206_WITHIN_14D_AFTER_WINDOW"
enriched[TARGET] = y
print({"rows": len(y), "positives": int(y.sum()), "prevalence": float(y.mean())})

[TARGET 1/107,966] matching partitions=0
[TARGET 10,000/107,966] matching partitions=400
[TARGET 20,000/107,966] matching partitions=860
[TARGET 30,000/107,966] matching partitions=1,314
[TARGET 40,000/107,966] matching partitions=1,735
[TARGET 50,000/107,966] matching partitions=2,132
[TARGET 60,000/107,966] matching partitions=2,521
[TARGET 70,000/107,966] matching partitions=2,944
[TARGET 80,000/107,966] matching partitions=3,370
[TARGET 90,000/107,966] matching partitions=3,802
[TARGET 100,000/107,966] matching partitions=4,218
[TARGET 107,966/107,966] matching partitions=4,578
{'rows': 1669294, 'positives': 87715, 'prevalence': 0.05254616622356517}


/tmp/ipykernel_187330/373049818.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched[TARGET] = y


## 6. Resolve benchmark variants

Each ranking artifact becomes a benchmark variant. Duplicate feature lists are deduplicated by content hash. Missing features are reported, and variants below the coverage threshold are skipped rather than silently altered.


In [6]:
# =============================================================================
# BUILD VARIANT MANIFESTS FROM RANKING FILES
# =============================================================================
import hashlib

variant_rows = []
variant_features = {}
seen_hashes = set()
for row in ranking_catalog.itertuples(index=False):
    ranking = pd.read_csv(row.path)
    ordered = ranking["feature"].dropna().astype(str).tolist()
    for cap in TOP_FEATURE_CAPS:
        selected = ordered[:cap]
        available = [c for c in selected if c in enriched.columns]
        missing = [c for c in selected if c not in enriched.columns]
        digest = hashlib.sha1("\n".join(available).encode("utf-8")).hexdigest()
        if digest in seen_hashes or not available:
            continue
        seen_hashes.add(digest)
        variant_id = re.sub(r"[^A-Za-z0-9_]+", "_", f"{row.experiment}__{Path(row.ranking_file).stem}__top{cap}")
        coverage = len(available) / max(1, len(selected))
        status = "READY" if coverage >= 0.90 else "SKIP_LOW_COVERAGE"
        variant_features[variant_id] = available
        variant_rows.append({
            "variant_id": variant_id,
            "experiment": row.experiment,
            "ranking_file": row.ranking_file,
            "requested_features": len(selected),
            "available_features": len(available),
            "missing_features": len(missing),
            "coverage": coverage,
            "status": status,
            "feature_hash": digest,
        })
variant_manifest = pd.DataFrame(variant_rows).sort_values(["status", "variant_id"]).reset_index(drop=True)
variant_manifest.to_csv(OUTPUT_ROOT / "variant_manifest.csv", index=False)
print(variant_manifest.to_string(index=False))

                                                                      variant_id                                 experiment                     ranking_file  requested_features  available_features  missing_features  coverage            status                             feature_hash
              hybrid_v2_cross_side_benchmark__feature_ranking_hybrid_v1__top1000             hybrid_v2_cross_side_benchmark    feature_ranking_hybrid_v1.csv                1000                 999                 1     0.999             READY d3846896fb6a5aa5e89a496287686cfe1ee5df29
               hybrid_v2_cross_side_benchmark__feature_ranking_hybrid_v1__top500             hybrid_v2_cross_side_benchmark    feature_ranking_hybrid_v1.csv                 500                 499                 1     0.998             READY 1b85467610b3321b237c076efc436aa5f4f4ffea
              hybrid_v2_cross_side_benchmark__feature_ranking_hybrid_v2__top1000             hybrid_v2_cross_side_benchmark    feature_ranking_hybri

## 7. Model-family registry

The core registry includes histogram gradient boosting, Extra Trees, Random Forest, elastic-net logistic regression, and SGD logistic regression. LightGBM, XGBoost, and CatBoost are added when installed.


In [7]:
# =============================================================================
# MODEL FAMILY REGISTRY
# =============================================================================
def build_model_registry():
    registry = {
        "hist_gradient_boosting": {
            "estimator": HistGradientBoostingClassifier(
                learning_rate=0.07, max_iter=150, max_leaf_nodes=31,
                min_samples_leaf=50, l2_regularization=1.0,
                early_stopping=True, validation_fraction=0.10,
                n_iter_no_change=15, random_state=RANDOM_STATE,
            ),
            "preprocess": "native_nan",
        },
        "extra_trees": {
            "estimator": ExtraTreesClassifier(
                n_estimators=300, max_depth=None, min_samples_leaf=20,
                max_features="sqrt", class_weight="balanced_subsample",
                n_jobs=N_JOBS, random_state=RANDOM_STATE,
            ),
            "preprocess": "median_impute",
        },
        "random_forest": {
            "estimator": RandomForestClassifier(
                n_estimators=300, max_depth=None, min_samples_leaf=20,
                max_features="sqrt", class_weight="balanced_subsample",
                n_jobs=N_JOBS, random_state=RANDOM_STATE,
            ),
            "preprocess": "median_impute",
        },
        "elastic_net_logistic": {
            "estimator": LogisticRegression(
                penalty="elasticnet", l1_ratio=0.20, solver="saga",
                C=0.5, max_iter=300, class_weight="balanced",
                n_jobs=N_JOBS, random_state=RANDOM_STATE,
            ),
            "preprocess": "median_scale",
        },
        "sgd_logistic": {
            "estimator": SGDClassifier(
                loss="log_loss", penalty="elasticnet", alpha=1e-5,
                l1_ratio=0.15, max_iter=1000, tol=1e-3,
                class_weight="balanced", random_state=RANDOM_STATE,
            ),
            "preprocess": "median_scale",
        },
    }
    if PACKAGE_AVAILABILITY["lightgbm"]:
        from lightgbm import LGBMClassifier
        registry["lightgbm"] = {
            "estimator": LGBMClassifier(
                n_estimators=500, learning_rate=0.05, num_leaves=31,
                subsample=0.8, colsample_bytree=0.8,
                class_weight="balanced", n_jobs=N_JOBS,
                random_state=RANDOM_STATE, verbosity=-1,
            ),
            "preprocess": "native_nan",
        }
    if PACKAGE_AVAILABILITY["xgboost"]:
        from xgboost import XGBClassifier
        registry["xgboost"] = {
            "estimator": XGBClassifier(
                n_estimators=500, learning_rate=0.05, max_depth=6,
                min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                eval_metric="logloss", tree_method="hist", n_jobs=N_JOBS,
                random_state=RANDOM_STATE,
            ),
            "preprocess": "native_nan",
        }
    if PACKAGE_AVAILABILITY["catboost"]:
        from catboost import CatBoostClassifier
        registry["catboost"] = {
            "estimator": CatBoostClassifier(
                iterations=500, learning_rate=0.05, depth=7,
                loss_function="Logloss", eval_metric="PRAUC",
                auto_class_weights="Balanced", random_seed=RANDOM_STATE,
                thread_count=N_JOBS, verbose=False,
            ),
            "preprocess": "median_impute",
        }
    return registry

MODEL_REGISTRY = build_model_registry()
print({"model_families": list(MODEL_REGISTRY), "count": len(MODEL_REGISTRY)})

{'model_families': ['hist_gradient_boosting', 'extra_trees', 'random_forest', 'elastic_net_logistic', 'sgd_logistic', 'xgboost'], 'count': 6}


## 8. Grouped benchmark execution

Results are written after every fold and model. Existing completed result files are reused. Training-row capping is stratified by class within each training fold; validation always uses every row.


In [8]:
# =============================================================================
# GROUPED MODEL-FAMILY BENCHMARK
# =============================================================================
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)): return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray): return json_safe(value.tolist())
    if isinstance(value, np.bool_): return bool(value)
    if isinstance(value, np.integer): return int(value)
    if isinstance(value, np.floating): return float(value)
    if isinstance(value, pd.Timestamp): return value.isoformat()
    if isinstance(value, Path): return str(value)
    if value is pd.NA: return None
    return value


def capped_training_indices(train_idx, target, cap, seed):
    if cap is None or len(train_idx) <= cap:
        return np.asarray(train_idx)
    rng = np.random.default_rng(seed)
    train_idx = np.asarray(train_idx)
    pos = train_idx[target[train_idx] == 1]
    neg = train_idx[target[train_idx] == 0]
    pos_take = min(len(pos), max(1, int(round(cap * len(pos) / len(train_idx)))))
    neg_take = min(len(neg), cap - pos_take)
    chosen = np.concatenate([
        rng.choice(pos, size=pos_take, replace=False),
        rng.choice(neg, size=neg_take, replace=False),
    ])
    rng.shuffle(chosen)
    return chosen


def make_estimator(spec, fold):
    estimator = clone(spec["estimator"])
    if hasattr(estimator, "random_state"):
        try: estimator.set_params(random_state=RANDOM_STATE + fold)
        except Exception: pass
    mode = spec["preprocess"]
    if mode == "native_nan":
        return estimator
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if mode == "median_scale":
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)

# Group all rows from the same aircraft; fall back to engine where aircraft is missing.
if "aircraft_identifier" in enriched.columns and enriched["aircraft_identifier"].notna().any():
    groups = enriched["aircraft_identifier"].astype("string").fillna(enriched["engine_serial"].astype(str))
else:
    groups = enriched["engine_serial"].astype(str)
target = enriched[TARGET].to_numpy(dtype=np.int8)
positive_groups = int(pd.DataFrame({"g": groups, "y": target}).groupby("g")["y"].max().sum())
n_splits = min(N_SPLITS, positive_groups)
if n_splits < 2:
    raise RuntimeError("Need at least two positive groups")
splits = list(GroupKFold(n_splits=n_splits).split(enriched, target, groups))

all_fold_rows = []
ready_variants = variant_manifest.loc[variant_manifest["status"].eq("READY"), "variant_id"].tolist()
for variant_number, variant_id in enumerate(ready_variants, 1):
    features = variant_features[variant_id]
    X = enriched[features].apply(pd.to_numeric, errors="coerce").astype("float32")
    print(f"[VARIANT {variant_number}/{len(ready_variants)}] {variant_id}: {len(features)} features")
    for model_name, spec in MODEL_REGISTRY.items():
        result_file = RESULT_ROOT / f"{variant_id}__{model_name}.json"
        if result_file.is_file():
            try:
                prior = json.loads(result_file.read_text())
                if prior.get("status") == "PASS":
                    all_fold_rows.extend(prior.get("folds", []))
                    print(f"  [REUSE] {model_name}")
                    continue
            except Exception:
                pass
        model_result = {"variant_id": variant_id, "model_family": model_name, "status": "FAIL", "folds": []}
        started = time.time()
        try:
            oof = np.full(len(enriched), np.nan, dtype=float)
            for fold, (train_idx, valid_idx) in enumerate(splits, 1):
                fit_idx = capped_training_indices(train_idx, target, TRAIN_ROW_CAP, RANDOM_STATE + fold)
                estimator = make_estimator(spec, fold)
                fit_kwargs = {}
                # HGB and boosting libraries receive balanced sample weights; sklearn forests/logistic already use class weights.
                if model_name in {"hist_gradient_boosting", "lightgbm", "xgboost", "catboost"}:
                    fit_kwargs["sample_weight"] = compute_sample_weight(class_weight="balanced", y=target[fit_idx])
                estimator.fit(X.iloc[fit_idx], target[fit_idx], **fit_kwargs)
                if hasattr(estimator, "predict_proba"):
                    probability = estimator.predict_proba(X.iloc[valid_idx])[:, 1]
                else:
                    score = estimator.decision_function(X.iloc[valid_idx])
                    probability = 1.0 / (1.0 + np.exp(-np.clip(score, -30, 30)))
                oof[valid_idx] = probability
                pr = float(average_precision_score(target[valid_idx], probability))
                roc = float(roc_auc_score(target[valid_idx], probability))
                prevalence = float(target[valid_idx].mean())
                fold_row = {
                    "variant_id": variant_id, "model_family": model_name, "fold": int(fold),
                    "train_rows_available": int(len(train_idx)), "train_rows_used": int(len(fit_idx)),
                    "valid_rows": int(len(valid_idx)), "features": int(len(features)),
                    "roc_auc": roc, "pr_auc": pr, "prevalence": prevalence,
                    "pr_lift": float(pr / prevalence),
                }
                model_result["folds"].append(fold_row)
                print(f"    fold {fold}: ROC={roc:.6f} PR={pr:.6f}")
                del estimator
                gc.collect()
            valid = np.isfinite(oof)
            model_result.update({
                "status": "PASS",
                "rows": int(valid.sum()),
                "features": int(len(features)),
                "roc_auc": float(roc_auc_score(target[valid], oof[valid])),
                "pr_auc": float(average_precision_score(target[valid], oof[valid])),
                "prevalence": float(target[valid].mean()),
                "elapsed_seconds": float(time.time() - started),
            })
            model_result["pr_lift"] = float(model_result["pr_auc"] / model_result["prevalence"])
            all_fold_rows.extend(model_result["folds"])
        except Exception:
            model_result["error"] = traceback.format_exc(limit=12)
            model_result["elapsed_seconds"] = float(time.time() - started)
        result_file.write_text(json.dumps(json_safe(model_result), indent=2), encoding="utf-8")
        print(f"  [{model_result['status']}] {model_name}")
    del X
    gc.collect()

pd.DataFrame(all_fold_rows).to_csv(OUTPUT_ROOT / "all_fold_results.csv", index=False)
print("PASS: model-family execution loop completed")

[VARIANT 1/28] hybrid_v2_cross_side_benchmark__feature_ranking_hybrid_v1__top1000: 999 features
    fold 1: ROC=0.685808 PR=0.121868
    fold 2: ROC=0.666578 PR=0.107650
    fold 3: ROC=0.699569 PR=0.109952
    fold 4: ROC=0.671917 PR=0.108296
    fold 5: ROC=0.681293 PR=0.109944
  [PASS] hist_gradient_boosting
    fold 1: ROC=0.686187 PR=0.115007
    fold 2: ROC=0.670066 PR=0.097169
    fold 3: ROC=0.665165 PR=0.094325
    fold 4: ROC=0.651340 PR=0.089577
    fold 5: ROC=0.666820 PR=0.103049
  [PASS] extra_trees
    fold 1: ROC=0.679286 PR=0.118475
    fold 2: ROC=0.667821 PR=0.101170
    fold 3: ROC=0.668757 PR=0.094758
    fold 4: ROC=0.651374 PR=0.094079
    fold 5: ROC=0.669074 PR=0.109226
  [PASS] random_forest
    fold 1: ROC=0.690710 PR=0.112989
    fold 2: ROC=0.669673 PR=0.094215
    fold 3: ROC=0.670684 PR=0.096534
    fold 4: ROC=0.653097 PR=0.090298
    fold 5: ROC=0.670611 PR=0.103282
  [PASS] elastic_net_logistic
    fold 1: ROC=0.505212 PR=0.059880
    fold 2: ROC=0.642

OSError: [Errno 28] No space left on device

In [13]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

BENCHMARK_ROOT = Path(
    "/raid3/e296408/All_ECFRs_working/"
    "ECFR_11002BC_ALL_ENGINE_REVISED_OPTIMIZED/"
    "data_richness/run_20260920_120031Z/"
    "hybrid_all_variants_model_family_benchmark_ver01"
)

PER_MODEL_ROOT = BENCHMARK_ROOT / "per_model_results"

MODEL_FAMILIES = [
    "hist_gradient_boosting",
    "extra_trees",
    "random_forest",
    "elastic_net_logistic",
    "sgd_logistic",
    "xgboost",
]

EXPECTED_VARIANTS = 28
EXPECTED_MODELS = len(MODEL_FAMILIES)
EXPECTED_RESULTS = EXPECTED_VARIANTS * EXPECTED_MODELS
EXPECTED_FOLDS = 5


def atomic_parquet(frame, destination):
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        destination.name + ".tmp"
    )

    frame.to_parquet(
        temporary,
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

    temporary.replace(destination)


def atomic_json(payload, destination):
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = destination.with_name(
        destination.name + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    temporary.replace(destination)


def parse_variant_from_filename(path, model_family):
    stem = Path(path).stem

    model_suffixes = [
        f"__{model_family}",
        f"_{model_family}",
    ]

    for suffix in model_suffixes:
        if stem.endswith(suffix):
            return stem[:-len(suffix)]

    # Fallback when filenames contain an extra result suffix.
    pattern = re.compile(
        rf"^(.*?)(?:__|_){re.escape(model_family)}"
        rf"(?:__.*|_result.*)?$"
    )

    match = pattern.match(stem)

    if match:
        return match.group(1)

    raise ValueError(
        f"Cannot parse variant from {path.name} "
        f"using model family {model_family}"
    )


json_files = sorted(
    PER_MODEL_ROOT.rglob("*.json")
)

if len(json_files) != EXPECTED_RESULTS:
    raise RuntimeError(
        f"Expected {EXPECTED_RESULTS} JSON checkpoints, "
        f"found {len(json_files)}"
    )

checkpoint_rows = []
fold_rows = []
payload_rows = []

for path in json_files:
    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    model_family = (
        payload.get("model_family")
        or payload.get("model_name")
        or payload.get("model")
    )

    if model_family not in MODEL_FAMILIES:
        raise ValueError(
            f"Unrecognized model family in {path}: "
            f"{model_family}"
        )

    variant = parse_variant_from_filename(
        path,
        model_family,
    )

    status = payload.get("status")
    folds = payload.get("folds")

    if status != "PASS":
        raise RuntimeError(
            f"Checkpoint is not PASS: {path}, "
            f"status={status}"
        )

    if not isinstance(folds, list):
        raise TypeError(
            f"folds is not a list in {path}"
        )

    if len(folds) != EXPECTED_FOLDS:
        raise RuntimeError(
            f"Expected {EXPECTED_FOLDS} folds in {path}, "
            f"found {len(folds)}"
        )

    checkpoint_rows.append(
        {
            "variant": variant,
            "model_family": model_family,
            "status": status,
            "checkpoint_path": str(path),
            "checkpoint_bytes": path.stat().st_size,
            "features": payload.get("features"),
            "elapsed_seconds":
                payload.get("elapsed_seconds"),
            "fold_count": len(folds),
        }
    )

    # Retain other scalar metadata from the payload.
    scalar_payload = {
        key: value
        for key, value in payload.items()
        if not isinstance(
            value,
            (list, dict),
        )
    }

    payload_rows.append(
        {
            "variant": variant,
            "model_family": model_family,
            **scalar_payload,
        }
    )

    for position, fold in enumerate(folds, 1):
        if not isinstance(fold, dict):
            raise TypeError(
                f"Fold {position} in {path} "
                "is not a dictionary"
            )

        fold_number = (
            fold.get("fold")
            or fold.get("fold_number")
            or fold.get("fold_id")
            or position
        )

        roc_auc = (
            fold.get("roc_auc")
            or fold.get("roc")
            or fold.get("validation_roc_auc")
        )

        pr_auc = (
            fold.get("pr_auc")
            or fold.get("average_precision")
            or fold.get("validation_pr_auc")
        )

        fold_rows.append(
            {
                "variant": variant,
                "model_family": model_family,
                "fold": int(fold_number),
                "roc_auc": roc_auc,
                "pr_auc": pr_auc,
                "checkpoint_path": str(path),
                **{
                    key: value
                    for key, value in fold.items()
                    if key not in {
                        "fold",
                        "fold_number",
                        "fold_id",
                        "roc_auc",
                        "roc",
                        "validation_roc_auc",
                        "pr_auc",
                        "average_precision",
                        "validation_pr_auc",
                    }
                    and np.isscalar(value)
                },
            }
        )


checkpoint_table = pd.DataFrame(
    checkpoint_rows
)

fold_table = pd.DataFrame(
    fold_rows
)

payload_table = pd.DataFrame(
    payload_rows
)

In [14]:
duplicate_pairs = checkpoint_table.duplicated(
    ["variant", "model_family"],
    keep=False,
)

if duplicate_pairs.any():
    display(
        checkpoint_table.loc[
            duplicate_pairs
        ].sort_values(
            ["variant", "model_family"]
        )
    )

    raise RuntimeError(
        "Duplicate variant-model checkpoint keys found"
    )

variant_count = (
    checkpoint_table["variant"].nunique()
)

model_count = (
    checkpoint_table["model_family"].nunique()
)

pair_count = len(
    checkpoint_table[
        ["variant", "model_family"]
    ].drop_duplicates()
)

fold_pair_count = len(
    fold_table[
        ["variant", "model_family", "fold"]
    ].drop_duplicates()
)

expected_fold_rows = (
    EXPECTED_RESULTS * EXPECTED_FOLDS
)

validation = pd.DataFrame(
    [
        {
            "check": "checkpoint_count",
            "passed":
                len(checkpoint_table)
                == EXPECTED_RESULTS,
            "detail":
                f"{len(checkpoint_table)}"
                f"/{EXPECTED_RESULTS}",
        },
        {
            "check": "variant_count",
            "passed":
                variant_count
                == EXPECTED_VARIANTS,
            "detail":
                f"{variant_count}"
                f"/{EXPECTED_VARIANTS}",
        },
        {
            "check": "model_family_count",
            "passed":
                model_count
                == EXPECTED_MODELS,
            "detail":
                f"{model_count}"
                f"/{EXPECTED_MODELS}",
        },
        {
            "check": "unique_variant_model_pairs",
            "passed":
                pair_count
                == EXPECTED_RESULTS,
            "detail":
                f"{pair_count}"
                f"/{EXPECTED_RESULTS}",
        },
        {
            "check": "fold_result_count",
            "passed":
                len(fold_table)
                == expected_fold_rows,
            "detail":
                f"{len(fold_table)}"
                f"/{expected_fold_rows}",
        },
        {
            "check": "unique_fold_keys",
            "passed":
                fold_pair_count
                == expected_fold_rows,
            "detail":
                f"{fold_pair_count}"
                f"/{expected_fold_rows}",
        },
        {
            "check": "all_status_pass",
            "passed":
                checkpoint_table["status"]
                .eq("PASS")
                .all(),
            "detail":
                checkpoint_table["status"]
                .value_counts()
                .to_dict(),
        },
    ]
)

display(validation)

if not validation["passed"].all():
    raise RuntimeError(
        "Checkpoint validation failed"
    )

,check,passed,detail
0,checkpoint_count,True,168/168
1,variant_count,True,28/28
2,model_family_count,True,6/6
3,unique_variant_model_pairs,True,168/168
4,fold_result_count,True,840/840
5,unique_fold_keys,True,840/840
6,all_status_pass,True,{'PASS': 168}


In [16]:
for column in ["roc_auc", "pr_auc"]:
    fold_table[column] = pd.to_numeric(
        fold_table[column],
        errors="coerce",
    )

if fold_table[
    ["roc_auc", "pr_auc"]
].isna().any().any():
    display(
        fold_table.loc[
            fold_table[
                ["roc_auc", "pr_auc"]
            ].isna().any(axis=1)
        ]
    )

    raise RuntimeError(
        "Some fold metrics could not be resolved"
    )

leaderboard = (
    fold_table
    .groupby(
        ["variant", "model_family"],
        as_index=False,
    )
    .agg(
        folds=("fold", "nunique"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        min_roc_auc=("roc_auc", "min"),
        max_roc_auc=("roc_auc", "max"),
        mean_pr_auc=("pr_auc", "mean"),
        std_pr_auc=("pr_auc", "std"),
        min_pr_auc=("pr_auc", "min"),
        max_pr_auc=("pr_auc", "max"),
    )
)

leaderboard["roc_rank"] = (
    leaderboard["mean_roc_auc"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

leaderboard["pr_rank"] = (
    leaderboard["mean_pr_auc"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

leaderboard["combined_rank"] = (
    leaderboard[
        ["roc_rank", "pr_rank"]
    ]
    .mean(axis=1)
)

leaderboard = leaderboard.sort_values(
    [
        "combined_rank",
        "mean_pr_auc",
        "mean_roc_auc",
    ],
    ascending=[
        True,
        False,
        False,
    ],
).reset_index(drop=True)

leaderboard["overall_rank"] = (
    np.arange(
        1,
        len(leaderboard) + 1,
    )
)

display(
    leaderboard.head(30)
)

,variant,model_family,folds,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,mean_pr_auc,std_pr_auc,min_pr_auc,max_pr_auc,roc_rank,pr_rank,combined_rank,overall_rank
0,hybrid_v3_2_all_fault_landing_memory_ver05__fe...,hist_gradient_boosting,5,0.745318,0.026570,0.703665,0.768620,0.125843,0.016350,0.101723,0.147600,1,1,1.0,1
1,hybrid_v4b_2_8004_all_fault_memory_ver04__feat...,hist_gradient_boosting,5,0.743585,0.029229,0.699287,0.772070,0.124751,0.016635,0.102500,0.148740,2,2,2.0,2
2,hybrid_v4b_2_8004_all_fault_memory_ver04__feat...,xgboost,5,0.734395,0.028413,0.689243,0.763421,0.124617,0.016825,0.099670,0.146997,13,3,8.0,3
3,hybrid_v3_2_all_fault_landing_memory_ver05__fe...,xgboost,5,0.735565,0.027933,0.693452,0.763951,0.120401,0.017411,0.093016,0.141280,12,7,9.5,4
4,hybrid_v4b_2_8004_all_fault_memory_ver04__feat...,extra_trees,5,0.740275,0.017679,0.712970,0.759800,0.119488,0.013717,0.098868,0.135818,5,15,10.0,5
5,hybrid_v4b_2_8004_all_fault_memory_ver04__feat...,random_forest,5,0.743261,0.015643,0.716550,0.754336,0.118785,0.013682,0.099157,0.136659,3,24,13.5,6
6,hybrid_v3_2_all_fault_landing_memory_ver05__fe...,xgboost,5,0.731906,0.012976,0.717985,0.751377,0.122048,0.011547,0.109810,0.136081,24,4,14.0,7
7,hybrid_v3_2_all_fault_landing_memory_ver05__fe...,random_forest,5,0.731386,0.018258,0.700173,0.744040,0.120716,0.015284,0.104404,0.145543,26,6,16.0,8
8,hybrid_v3_fault_history_benchmark__feature_ran...,random_forest,5,0.732622,0.015291,0.706909,0.744459,0.119584,0.013582,0.108258,0.142608,19,13,16.0,9
9,hybrid_v3_fault_history_benchmark__feature_ran...,xgboost,5,0.730969,0.014893,0.709721,0.750248,0.121738,0.010770,0.108470,0.135203,29,5,17.0,10


## 9. Consolidated leaderboard and stability report

The leaderboard ranks primarily by PR AUC, then ROC AUC. It includes mean and standard deviation across folds, training-row policy, feature count, and package availability.


In [17]:
# =============================================================================
# CONSOLIDATE RESTARTABLE RESULT FILES
# =============================================================================
model_rows = []
fold_rows = []
for path in sorted(RESULT_ROOT.glob("*.json")):
    try:
        result = json.loads(path.read_text())
    except Exception:
        continue
    for row in result.get("folds", []):
        fold_rows.append(row)
    model_rows.append({
        "variant_id": result.get("variant_id"),
        "model_family": result.get("model_family"),
        "status": result.get("status"),
        "features": result.get("features"),
        "rows": result.get("rows"),
        "roc_auc": result.get("roc_auc"),
        "pr_auc": result.get("pr_auc"),
        "prevalence": result.get("prevalence"),
        "pr_lift": result.get("pr_lift"),
        "elapsed_seconds": result.get("elapsed_seconds"),
        "error": result.get("error", ""),
    })
leaderboard = pd.DataFrame(model_rows)
fold_frame = pd.DataFrame(fold_rows)
if not fold_frame.empty:
    stability = fold_frame.groupby(["variant_id", "model_family"]).agg(
        fold_roc_mean=("roc_auc", "mean"), fold_roc_std=("roc_auc", "std"),
        fold_pr_mean=("pr_auc", "mean"), fold_pr_std=("pr_auc", "std"),
        folds=("fold", "nunique"), train_rows_used=("train_rows_used", "mean"),
    ).reset_index()
    leaderboard = leaderboard.merge(stability, on=["variant_id", "model_family"], how="left")
leaderboard = leaderboard.sort_values(["status", "pr_auc", "roc_auc"], ascending=[True, False, False], na_position="last")
leaderboard.to_csv(OUTPUT_ROOT / "model_family_leaderboard.csv", index=False)
fold_frame.to_csv(OUTPUT_ROOT / "model_family_fold_results.csv", index=False)

passed = leaderboard.loc[leaderboard["status"].eq("PASS")].copy()
summary = {
    "benchmark_contract": "HYBRID_ALL_VARIANTS_MODEL_FAMILY_BENCHMARK_VER01",
    "part_b_run_root": str(PART_B_RUN_ROOT),
    "variants_ready": int(len(ready_variants)),
    "model_families": list(MODEL_REGISTRY),
    "completed_runs": int(len(passed)),
    "failed_runs": int((leaderboard["status"] == "FAIL").sum()),
    "train_row_cap": TRAIN_ROW_CAP,
    "folds": n_splits,
    "best_by_pr_auc": json_safe(passed.iloc[0].to_dict()) if len(passed) else None,
    "best_by_roc_auc": json_safe(passed.sort_values("roc_auc", ascending=False).iloc[0].to_dict()) if len(passed) else None,
    "method_note": "Variant feature lists inherit the screening methodology of their source Hybrid experiment.",
}
summary_path = OUTPUT_ROOT / "model_family_benchmark_summary.json"
summary_path.write_text(json.dumps(json_safe(summary), indent=2), encoding="utf-8")
with summary_path.open("r", encoding="utf-8") as handle:
    json.load(handle)

print("\nTOP 20 BY PR AUC")
print(passed.head(20).to_string(index=False))
print("\nBEST BY ROC AUC")
print(passed.sort_values("roc_auc", ascending=False).head(10).to_string(index=False))
print(json.dumps(json_safe(summary), indent=2))
print("PASS: consolidated leaderboard and summary written")


TOP 20 BY PR AUC
                                                                      variant_id           model_family status  features    rows  roc_auc   pr_auc  prevalence  pr_lift  elapsed_seconds error  fold_roc_mean  fold_roc_std  fold_pr_mean  fold_pr_std  folds  train_rows_used
hybrid_v3_2_all_fault_landing_memory_ver05__feature_ranking_hybrid_v3_2__top1000 hist_gradient_boosting   PASS       995 1669294 0.744626 0.122899    0.052546 2.338876        48.393879             0.745318      0.026570      0.125843     0.016350      5         500000.0
 hybrid_v4b_2_8004_all_fault_memory_ver04__feature_ranking_hybrid_v4b_2__top1000                xgboost   PASS       969 1669294 0.733888 0.122598    0.052546 2.333142        77.840281             0.734395      0.028413      0.124617     0.016825      5         500000.0
 hybrid_v4b_2_8004_all_fault_memory_ver04__feature_ranking_hybrid_v4b_2__top1000 hist_gradient_boosting   PASS       969 1669294 0.742790 0.121903    0.052546 2.319931  